# Lab 8: Image Segmentation 1

*Written by Amirhossein Alimohammadi, modified by Yalda.*

In Assignment 2 your U-Net gave every pixel of a cell image one of **2** labels: cell or background.
Today we segment **street scenes** into **19 classes** (road, car, person, sky, ...): this is what a self-driving car
has to see. You will find out why "95% of pixels correct" can be a bad model, build **SegNet** (a cousin of U-Net)
on top of a pretrained VGG16, train it, and at the end use a pretrained model for a "portrait mode" photo effect.

1. **The data.** Cityscapes street scenes and their label maps.
2. **Measuring a segmentation.** Pixel accuracy, IoU and mIoU, and two "cheating" baselines.
3. **SegNet's trick.** Max-unpooling: remember *where* the maximum was.
4. **Build SegNet.** A pretrained VGG16 encoder and a mirrored decoder.
5. **Train it.** Augmenting an image *and* its mask.
6. **Results.** Which classes are easy, which are hard, and what gets confused.
7. **Segmentation in the wild.** A pretrained model and portrait mode on a photo.

**Extras** (in class if you have time, otherwise at home): A. fixing the rare classes with class weights and a Dice
loss, B. counting cars with flood fill.

| Part | Time |
|---|---|
| Setup + 1. The data | 15 min |
| 2. Measuring a segmentation | 25 min |
| 3. SegNet's trick | 15 min |
| 4. Build SegNet | 20 min |
| 5. Train it | 25 min |
| 6. Results | 15 min |
| 7. Segmentation in the wild | 15 min |
| Extra A. Fixing the rare classes | 35 min |
| Extra B. How many cars? | 20 min |

For each **TODO**, write your code, then run the check cell. The solutions are hidden; open them only after you try.
Use a GPU for Parts 4 to 6 (in Colab: *Runtime → Change runtime type → T4 GPU*). On a CPU, training takes hours.

## Environment (skip this on Colab)

On Colab everything is already installed. To run this notebook on your own machine or a lab machine, create a conda
environment **once** from a terminal:

```bash
conda create -n vc python=3.11 -y && conda activate vc && pip install torch torchvision numpy matplotlib pillow kagglehub jupyter ipykernel && python -m ipykernel install --user --name vc
```

Then pick the **vc** kernel in Jupyter / VS Code (top right). On Windows with an NVIDIA GPU, get the install command
from [pytorch.org](https://pytorch.org/get-started/locally/) instead.

If you already have an environment, the next cell installs anything missing into the kernel you are using.
If it installs something, **restart the kernel** (*Kernel → Restart*), then continue.

In [ ]:
%pip install -q torch torchvision numpy matplotlib pillow kagglehub

In [ ]:
# check your environment
import sys
print("python:", sys.version.split()[0], " from", sys.executable)
try:
    import torch
    import torchvision
    import kagglehub
    print("torch:", torch.__version__, " torchvision:", torchvision.__version__)
    print("NVIDIA GPU:", torch.cuda.is_available(), " Apple GPU:", torch.backends.mps.is_available())
except ImportError as e:
    print("ERROR:", e)
    print("Restart the kernel and run this cell again, or pick the vc kernel (see above).")

## Setup

In [ ]:
import os
import time
import random
import urllib.request

import numpy as np
import matplotlib.pyplot as plt
from PIL import Image, ImageFilter

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader
from torchvision.models import vgg16_bn, VGG16_BN_Weights
import kagglehub

if torch.cuda.is_available():
    device = "cuda"
elif torch.backends.mps.is_available():   # Apple GPU
    device = "mps"
else:
    device = "cpu"
print("device:", device)

## 1. The data

[Cityscapes](https://www.cityscapes-dataset.com/) ([Cordts et al., 2016](https://arxiv.org/abs/1604.01685)) has
5,000 photos taken from a car driving through 50 German cities. People drew the outline of every object by hand,
about **1.5 hours per image**. We use a small version from Kaggle: the photos are resized to **128×256** pixels
(642 MB, about a minute to download):

| Split | Images | We use it for |
|---|---|---|
| `train` | 2,975 | training (2,675) and validation (300) |
| `val` | 500 | our **test** set (the real Cityscapes test labels are secret) |

Each photo comes with a **label map**: a 128×256 array with one class number (0 to 18) per pixel.
Pixels nobody labeled (the car's own hood, the image border, ...) are `-1`; we change them to **255** and tell the
loss and the metrics to **ignore** them.

In [ ]:
path = kagglehub.dataset_download("sakshaymahna/cityscapes-depth-and-segmentation")
print("downloaded to:", path)


def load_split(split):
    folder = os.path.join(path, "data", split)
    n = len(os.listdir(os.path.join(folder, "image")))
    images = np.zeros((n, 128, 256, 3), dtype=np.uint8)
    labels = np.zeros((n, 128, 256), dtype=np.uint8)
    for i in range(n):
        img = np.load(os.path.join(folder, "image", f"{i}.npy"))   # floats 0..1, shape (128, 256, 3)
        lab = np.load(os.path.join(folder, "label", f"{i}.npy"))   # floats -1, 0, 1, ..., 18, shape (128, 256)
        lab[lab < 0] = 255                                          # "no label" -> 255
        images[i] = np.round(img * 255)
        labels[i] = lab
    return torch.from_numpy(images), torch.from_numpy(labels)


start = time.time()
train_images, train_labels = load_split("train")
test_images, test_labels = load_split("val")
print(f"loaded in {time.time() - start:.0f} s")
print("train:", tuple(train_images.shape), train_images.dtype, " labels:", tuple(train_labels.shape), train_labels.dtype)
print("test: ", tuple(test_images.shape))
print("label values in image 0:", torch.unique(train_labels[0]).tolist())

In [ ]:
CLASSES = ["road", "sidewalk", "building", "wall", "fence", "pole", "traffic light", "traffic sign",
           "vegetation", "terrain", "sky", "person", "rider", "car", "truck", "bus", "train",
           "motorcycle", "bicycle"]
NUM_CLASSES = len(CLASSES)
IGNORE = 255

# the official Cityscapes color of every class (R, G, B)
COLORS = np.array([
    [128, 64, 128], [244, 35, 232], [70, 70, 70], [102, 102, 156], [190, 153, 153], [153, 153, 153],
    [250, 170, 30], [220, 220, 0], [107, 142, 35], [152, 251, 152], [70, 130, 180], [220, 20, 60],
    [255, 0, 0], [0, 0, 142], [0, 0, 70], [0, 60, 100], [0, 80, 100], [0, 0, 230], [119, 11, 32]], dtype=np.uint8)

# the legend
fig, ax = plt.subplots(figsize=(14, 1.3))
ax.imshow(COLORS[None])
ax.set_xticks(range(NUM_CLASSES))
ax.set_xticklabels([f"{i} {c}" for i, c in enumerate(CLASSES)], rotation=45, ha="right", fontsize=9)
ax.set_yticks([])
plt.show()

A label map is just numbers, so `plt.imshow(label)` gives a picture that is hard to read. We give every class its
own color instead.

**TODO 1:** write `colorize(label)`. Start with a black RGB image. For every class `c`, find the pixels
where `label == c` (a True/False **mask**) and paint them with `COLORS[c]`. Pixels with 255 stay black.

In [ ]:
def colorize(label):
    # label: (H, W) array of class numbers. Returns an (H, W, 3) uint8 RGB image
    label = np.asarray(label)
    H = label.shape[0]
    W = label.shape[1]
    colored = np.zeros((H, W, 3), dtype=np.uint8)
    for c in range(NUM_CLASSES):
        # TODO: paint the pixels of class c with COLORS[c]
        ...
    return colored

#### Solution

Try it yourself first, then open the hidden cell below.

In [ ]:
#@title Solution

def colorize(label):
    # label: (H, W) array of class numbers. Returns an (H, W, 3) uint8 RGB image
    label = np.asarray(label)
    H = label.shape[0]
    W = label.shape[1]
    colored = np.zeros((H, W, 3), dtype=np.uint8)
    for c in range(NUM_CLASSES):
        colored[label == c] = COLORS[c]
    return colored

In [ ]:
# run this cell to check your code
toy = np.array([[0, 13, 255],
                [10, 10, 0]])
out = colorize(toy)
print(out[0])
print("correct:", out.shape == (2, 3, 3) and out.dtype == np.uint8 and (out[0, 0] == COLORS[0]).all()
      and (out[0, 1] == COLORS[13]).all() and (out[0, 2] == 0).all() and (out[1, :2] == COLORS[10]).all())

In [ ]:
# photo, label map, and the label drawn on top of the photo (run again for other images)
fig, axes = plt.subplots(3, 3, figsize=(12, 6))
for row, i in enumerate(random.sample(range(len(train_images)), 3)):
    img = train_images[i].numpy()
    lab = colorize(train_labels[i])
    axes[row, 0].imshow(img)
    axes[row, 1].imshow(lab)
    axes[row, 2].imshow((0.5 * img + 0.5 * lab).astype(np.uint8))
    axes[row, 0].set_ylabel(f"image {i}")
for ax, t in zip(axes[0], ["photo", "label (black = no label)", "overlay"]):
    ax.set_title(t)
for ax in axes.flatten():
    ax.set_xticks([])
    ax.set_yticks([])
plt.tight_layout()
plt.show()

Like in Lab 7, we keep some training images aside as a **validation** set to watch during training, and use the test
set only once at the end.

In [ ]:
torch.manual_seed(0)
perm = torch.randperm(len(train_images))
val_idx = perm[:300]
tr_idx = perm[300:]
print("train:", len(tr_idx), " validation:", len(val_idx), " test:", len(test_images))

### Not all classes are equal

How many pixels of each class are there in the training set?

In [ ]:
counts = torch.bincount(train_labels.flatten(), minlength=256)[:NUM_CLASSES].double()
freq = (counts / counts.sum()).numpy()

plt.figure(figsize=(12, 3.5))
plt.bar(CLASSES, freq * 100, color=COLORS / 255, edgecolor="black")
plt.yscale("log")
plt.ylabel("% of labeled pixels (log scale)")
plt.xticks(rotation=45, ha="right")
plt.show()
for c in np.argsort(-freq):
    print(f"{CLASSES[c]:14s} {100 * freq[c]:6.2f} %")

**Think:** road and building together are more than half of all pixels. Motorcycles are about one pixel in a thousand.
What would a lazy model that only wants "most pixels correct" learn to do?

## 2. Measuring a segmentation

### The confusion matrix

Segmentation is classification of **every pixel**. So, like in Lab 5, we can count a **confusion matrix**: entry
`[t, p]` = how many pixels of true class `t` were predicted as class `p`. Correct pixels are on the diagonal.
All the metrics below come from this one table.

A fast way to count it: give every (true, predicted) pair its own number `t * n + p`, where `n` is the number of
classes. For `n = 3`:

| pair `(t, p)` | (0,0) | (0,1) | (0,2) | (1,0) | (1,1) | ... | (2,2) |
|---|---|---|---|---|---|---|---|
| `t * 3 + p` | 0 | 1 | 2 | 3 | 4 | ... | 8 |

`torch.bincount(x, minlength=n * n)` counts how often each number `0, 1, ..., n*n-1` appears in `x`.
`.reshape(n, n)` then puts the counts back into a table: number 4 lands in row 1, column 1.

**TODO 2:** write `confusion_matrix(pred, true, n)`:
1. keep only the pixels where `true != IGNORE` (index both tensors with that mask),
2. compute the pair numbers `true * n + pred`,
3. count them with `torch.bincount` and reshape to `(n, n)`.

In [ ]:
def confusion_matrix(pred, true, n=NUM_CLASSES):
    # pred, true: integer tensors of the same shape (any shape). Returns an (n, n) tensor of counts
    pred = pred.flatten().long()
    true = true.flatten().long()
    # TODO: keep only the labeled pixels
    keep = ...
    pred = ...
    true = ...
    # TODO: one number per (true, pred) pair, count them, reshape to (n, n)
    pairs = ...
    counts = ...
    return counts.reshape(n, n)

#### Solution

Try it yourself first, then open the hidden cell below.

In [ ]:
#@title Solution

def confusion_matrix(pred, true, n=NUM_CLASSES):
    # pred, true: integer tensors of the same shape (any shape). Returns an (n, n) tensor of counts
    pred = pred.flatten().long()
    true = true.flatten().long()
    keep = true != IGNORE
    pred = pred[keep]
    true = true[keep]
    pairs = true * n + pred
    counts = torch.bincount(pairs, minlength=n * n)
    return counts.reshape(n, n)

In [ ]:
# run this cell to check your code
true = torch.tensor([[0, 0, 1, 1],
                     [2, 2, 2, 255]])
pred = torch.tensor([[0, 1, 1, 1],
                     [2, 0, 2, 2]])
cm = confusion_matrix(pred, true, 3)
print(cm)
print("correct:", cm.tolist() == [[1, 1, 0], [0, 2, 0], [1, 0, 2]])

### Pixel accuracy, IoU and mIoU

- **Pixel accuracy** = correct pixels / all labeled pixels = diagonal sum / total sum.
- **IoU** of class $c$ (Intersection over Union, the same IoU as for boxes in Lab 6, but with pixels):

$$\text{IoU}_c = \frac{|\text{true}_c \cap \text{pred}_c|}{|\text{true}_c \cup \text{pred}_c|}
= \frac{\text{TP}_c}{\text{TP}_c + \text{FP}_c + \text{FN}_c}$$

  In the confusion matrix: $\text{TP}_c$ = `cm[c, c]`, the union = (row $c$ sum) + (column $c$ sum) − `cm[c, c]`
  (the diagonal is in both sums, so we subtract it once).
- **mIoU** (mean IoU) = the average IoU over the classes. Every class counts the same, big or small.
  This is the main number used to compare segmentation models.

**TODO 3:** write `metrics(cm)`. Use `cm.sum(dim=1)` for row sums and `cm.sum(dim=0)` for column sums, and
`torch.diag(cm)` for the diagonal. A class that is in neither the truth nor the prediction has union 0: give it
IoU `nan` (not a number) and use `torch.nanmean` so it is skipped in the average.

In [ ]:
def metrics(cm):
    # cm: (n, n) confusion matrix. Returns pixel accuracy, mIoU, and the IoU of every class
    cm = cm.double()
    tp = torch.diag(cm)
    # TODO: pixel accuracy
    acc = ...
    # TODO: union of every class (a tensor of n numbers)
    union = ...
    # TODO: IoU of every class (nan where union == 0), and their mean
    iou = ...
    miou = ...
    return acc.item(), miou.item(), iou

#### Solution

Try it yourself first, then open the hidden cell below.

In [ ]:
#@title Solution

def metrics(cm):
    # cm: (n, n) confusion matrix. Returns pixel accuracy, mIoU, and the IoU of every class
    cm = cm.double()
    tp = torch.diag(cm)
    acc = tp.sum() / cm.sum()
    union = cm.sum(dim=1) + cm.sum(dim=0) - tp
    iou = tp / union                     # 0 / 0 gives nan by itself
    iou[union == 0] = float("nan")
    miou = torch.nanmean(iou)
    return acc.item(), miou.item(), iou

In [ ]:
# run this cell to check your code (the same toy example as above)
acc, miou, iou = metrics(cm)
print(f"pixel accuracy {acc:.3f}   mIoU {miou:.3f}   IoU per class {iou.tolist()}")
cm4 = torch.zeros(4, 4, dtype=torch.long)
cm4[:3, :3] = cm                     # class 3 never appears
acc4, miou4, iou4 = metrics(cm4)
print("correct:", abs(acc - 5 / 7) < 1e-6 and np.allclose(iou.tolist(), [1 / 3, 2 / 3, 2 / 3])
      and abs(miou - 5 / 9) < 1e-6 and torch.isnan(iou4[3]).item() and abs(miou4 - 5 / 9) < 1e-6)

### Two models that "cheat"

Before training anything, let's see what numbers two silly models get on the test set.

1. **"Everything is road."** Predict class 0 for every pixel.
2. **"The average street."** Cityscapes photos are all taken from the same car, so the sky is almost always at the
   top and the road at the bottom. For every pixel **position**, predict the class that is most common at that
   position in the training set. This model never even looks at the photo.

In [ ]:
# "the average street": count every class at every pixel position over the training set
position_counts = torch.zeros(NUM_CLASSES, 128, 256)
for c in range(NUM_CLASSES):
    position_counts[c] = (train_labels[tr_idx] == c).sum(dim=0)
average_street = position_counts.argmax(dim=0)          # (128, 256)

all_road = torch.zeros_like(test_labels)
same_street = average_street.expand(len(test_labels), 128, 256)

results = {}
for name, pred in [("everything is road", all_road), ("the average street", same_street)]:
    acc, miou, iou = metrics(confusion_matrix(pred, test_labels))
    results[name] = (acc, miou)
    print(f"{name:20s}  pixel accuracy {acc:.3f}   mIoU {miou:.3f}")

from matplotlib.patches import Patch

plt.figure(figsize=(6, 3))
plt.imshow(colorize(average_street))
plt.title("the average street (the same prediction for every photo)")
plt.axis("off")
used = torch.unique(average_street).tolist()            # the classes that appear in this picture
legend = [Patch(color=COLORS[c] / 255, label=CLASSES[c]) for c in used]
plt.legend(handles=legend, bbox_to_anchor=(1.02, 1), loc="upper left")
plt.show()

**Think:** the average street gets more than half of the pixels right without looking at the photo. Why is its mIoU
so much lower than its pixel accuracy? Which classes does it never predict at all?

This is why papers report **mIoU** and not pixel accuracy. In medical images (like your cells in A2) people also use
the **Dice** score, $\frac{2\,|\text{true} \cap \text{pred}|}{|\text{true}| + |\text{pred}|}$. It is closely related:
$\text{Dice} = \frac{2\,\text{IoU}}{1 + \text{IoU}}$, so it ranks models the same way.

## 3. SegNet's trick: max-unpooling

Every segmentation network has the same problem. The **encoder** (a classification CNN like VGG or ResNet) makes the
image smaller and smaller with pooling: 128×256 → 4×8. That is good for understanding *what* is in the image, but we
need an answer for every pixel, so a **decoder** must make it big again. When it does, *where exactly* were the
edges?

```
             encoder (VGG16)                              decoder
 128x256 ─conv─ pool ─conv─ pool ─ ... ─ 4x8 ─ unpool ─conv─ unpool ─conv─ ... ─ 128x256 x 19 scores
               │                                   ▲
               └────── where was each maximum? ────┘   SegNet: only the positions (indices)
               └────── the whole feature map  ─────┘   U-Net (A2): copy and concatenate everything
```

**SegNet** ([Badrinarayanan et al., 2017](https://arxiv.org/abs/1511.00561)) has a cheap answer. Each 2×2 max-pool
keeps one number out of four. SegNet **remembers which one** it was (the **index**). On the way back up,
**max-unpooling** puts every value back at exactly that position and fills the other three with 0.
The convolutions after it then fill in the holes. Remembering a position is much cheaper than U-Net's copy of the
whole feature map.

PyTorch does this with `return_indices=True`:

In [ ]:
x = torch.tensor([[1., 2., 9., 0.],
                  [8., 3., 1., 4.],
                  [0., 6., 2., 2.],
                  [5., 1., 7., 3.]]).view(1, 1, 4, 4)     # (batch, channels, H, W)

pooled, idx = F.max_pool2d(x, kernel_size=2, stride=2, return_indices=True)
print("pooled:\n", pooled[0, 0])
print("indices:\n", idx[0, 0])
print("unpooled:\n", F.max_unpool2d(pooled, idx, kernel_size=2, stride=2)[0, 0])

The index is the position in the **flattened** 4×4 input: 9 was at row 0, column 2, so its index is
`0 * 4 + 2 = 2`. And 8 was at row 1, column 0: `1 * 4 + 0 = 4`.

**TODO 4:** write max-unpooling yourself for one channel: make a flat array of zeros of size `H_out * W_out`,
put every pooled value at its index, then reshape to `(H_out, W_out)`. Hint: with NumPy, `out[idx] = values` writes
many positions at once.

In [ ]:
def my_unpool(pooled, idx, H_out, W_out):
    # pooled, idx: (h, w) NumPy arrays. Returns an (H_out, W_out) array
    # TODO: a flat array of zeros
    out = ...
    # TODO: every value at its index (flatten pooled and idx first)
    ...
    return out.reshape(H_out, W_out)

#### Solution

Try it yourself first, then open the hidden cell below.

In [ ]:
#@title Solution

def my_unpool(pooled, idx, H_out, W_out):
    # pooled, idx: (h, w) NumPy arrays. Returns an (H_out, W_out) array
    out = np.zeros(H_out * W_out)
    out[idx.flatten()] = pooled.flatten()
    return out.reshape(H_out, W_out)

In [ ]:
# run this cell to check your code, on a random 8x8 image
x = torch.rand(1, 1, 8, 8)
pooled, idx = F.max_pool2d(x, 2, 2, return_indices=True)
mine = my_unpool(pooled[0, 0].numpy(), idx[0, 0].numpy(), 8, 8)
print("correct:", np.allclose(mine, F.max_unpool2d(pooled, idx, 2, 2)[0, 0].numpy()))

For SegNet we need the same thing for a whole batch of feature maps `(batch, channels, h, w)` on the GPU.
It is your `my_unpool`, with `out.scatter(2, idx, values)`, which is PyTorch's way to write `out[idx] = values`
along the last axis. (PyTorch also has `F.max_unpool2d`, but older versions don't have it on Apple GPUs.)

In [ ]:
def max_unpool(x, idx):
    # x, idx: (batch, channels, h, w). Returns (batch, channels, 2h, 2w)
    N, C, h, w = x.shape
    out = torch.zeros(N, C, 2 * h * 2 * w, device=x.device, dtype=x.dtype)   # flat zeros for every channel
    out = out.scatter(2, idx.flatten(2), x.flatten(2))                       # every value at its index
    return out.view(N, C, 2 * h, 2 * w)


x = torch.rand(2, 3, 8, 8)
pooled, idx = F.max_pool2d(x, 2, 2, return_indices=True)
print("same as F.max_unpool2d:", torch.equal(max_unpool(pooled, idx), F.max_unpool2d(pooled, idx, 2, 2)))

In [ ]:
# make a street photo 2x smaller with max-pooling, then big again in three different ways
img = train_images[0].permute(2, 0, 1)[None].float() / 255      # (1, 3, 128, 256)
small, idx = F.max_pool2d(img, 2, 2, return_indices=True)       # (1, 3, 64, 128)
ways = {
    "original 128x256": img,
    "max-pooled: 64x128": small,
    "nearest upsampling": F.interpolate(small, scale_factor=2, mode="nearest"),
    "bilinear upsampling": F.interpolate(small, scale_factor=2, mode="bilinear"),
    "max-unpooling (SegNet)": max_unpool(small, idx),
}
fig, axes = plt.subplots(1, 5, figsize=(20, 2.6))
for ax, (name, t) in zip(axes, ways.items()):
    H = t.shape[2]
    zoom = t[0, :, H * 5 // 16:H * 13 // 16, H // 2:H * 3 // 2]    # the middle of the photo
    ax.imshow(zoom.permute(1, 2, 0).clamp(0, 1).numpy())
    ax.set_title(name + ", zoomed")
    ax.axis("off")
plt.show()

**Think:** the unpooled image is mostly black (zeros), but the dots that are left sit at **exactly** the right places,
for example along the lane markings. (The dots are colored because R, G and B are pooled separately, so each
channel keeps its own maximum.) Nearest and bilinear upsampling fill every pixel, but the edges are blocky or blurry.
In SegNet, the convolutions after each unpool learn to fill in the black holes.

## 4. Build SegNet

**Encoder.** We take VGG16 (with batch norm), pretrained on ImageNet, from torchvision, but only its convolutional part
`vgg.features`. It is a list of 13 conv blocks (Conv2d + BatchNorm2d + ReLU) with 5 MaxPool2d layers in between.
We cut it at the pools into 5 **stages**, and do the pooling ourselves so we get the indices.
(The download is 528 MB, because it includes VGG's huge classification layers that we throw away.)

In [ ]:
vgg = vgg16_bn(weights=VGG16_BN_Weights.DEFAULT)
print(vgg.features[:7], "...")


def vgg_stages():
    stages = []
    current = []
    for layer in vgg16_bn(weights=VGG16_BN_Weights.DEFAULT).features:
        if isinstance(layer, nn.MaxPool2d):    # a pool ends the stage
            stages.append(nn.Sequential(*current))
            current = []
        else:
            current.append(layer)
    return nn.ModuleList(stages)


for s, stage in enumerate(vgg_stages()):
    convs = [m for m in stage if isinstance(m, nn.Conv2d)]
    print(f"stage {s}: {len(convs)} convs, {convs[0].in_channels} -> {convs[-1].out_channels} channels")

**Decoder.** The mirror image: 5 stages, each one is **unpool, then 2 conv blocks** (the conv blocks are the
same `Conv2d + BatchNorm2d + ReLU` as your `twoConvBlock` in A2). The channels go 512 → 512 → 256 → 128 → 64 → 64.
A last 3×3 conv gives **19 scores per pixel**, one per class, exactly like the 10 scores per image in Lab 5.

**TODO 5:** write `forward`:
1. **down:** for every encoder stage: run the stage, then `F.max_pool2d(x, 2, 2, return_indices=True)`, and append
   the indices to the list `indices`,
2. **up:** for every decoder stage, use the indices in **reverse** order (the last pool is undone first):
   `max_unpool(x, idx)`, then run the decoder stage,
3. return `self.classifier(x)`.

Hint: `zip(self.decoder, indices[::-1])` gives you each decoder stage with its matching indices.

In [ ]:
def conv_block(in_ch, out_ch):
    return nn.Sequential(
        nn.Conv2d(in_ch, out_ch, kernel_size=3, padding=1),
        nn.BatchNorm2d(out_ch),
        nn.ReLU(inplace=True),
    )


class SegNet(nn.Module):
    def __init__(self, num_classes=NUM_CLASSES):
        super().__init__()
        self.encoder = vgg_stages()                     # 5 pretrained stages
        channels = [(512, 512), (512, 256), (256, 128), (128, 64), (64, 64)]
        self.decoder = nn.ModuleList()
        for in_ch, out_ch in channels:
            self.decoder.append(nn.Sequential(conv_block(in_ch, in_ch), conv_block(in_ch, out_ch)))
        self.classifier = nn.Conv2d(64, num_classes, kernel_size=3, padding=1)

    def forward(self, x):
        # x: (batch, 3, H, W). Returns scores (batch, num_classes, H, W)
        indices = []
        # TODO: down. Every encoder stage, then a max-pool that also returns the indices
        ...
        # TODO: up. Unpool with the matching indices, then the decoder stage
        ...
        # TODO: 19 scores per pixel
        return ...

#### Solution

Try it yourself first, then open the hidden cell below.

In [ ]:
#@title Solution

def conv_block(in_ch, out_ch):
    return nn.Sequential(
        nn.Conv2d(in_ch, out_ch, kernel_size=3, padding=1),
        nn.BatchNorm2d(out_ch),
        nn.ReLU(inplace=True),
    )


class SegNet(nn.Module):
    def __init__(self, num_classes=NUM_CLASSES):
        super().__init__()
        self.encoder = vgg_stages()                     # 5 pretrained stages
        channels = [(512, 512), (512, 256), (256, 128), (128, 64), (64, 64)]
        self.decoder = nn.ModuleList()
        for in_ch, out_ch in channels:
            self.decoder.append(nn.Sequential(conv_block(in_ch, in_ch), conv_block(in_ch, out_ch)))
        self.classifier = nn.Conv2d(64, num_classes, kernel_size=3, padding=1)

    def forward(self, x):
        # x: (batch, 3, H, W). Returns scores (batch, num_classes, H, W)
        indices = []
        for stage in self.encoder:
            x = stage(x)
            x, idx = F.max_pool2d(x, 2, 2, return_indices=True)
            indices.append(idx)
        for stage, idx in zip(self.decoder, indices[::-1]):
            x = max_unpool(x, idx)
            x = stage(x)
        return self.classifier(x)

In [ ]:
# run this cell to check your code
model = SegNet().to(device)
x = torch.randn(2, 3, 128, 256, device=device)
with torch.no_grad():
    out = model(x)
n_enc = sum(p.numel() for p in model.encoder.parameters())
n_all = sum(p.numel() for p in model.parameters())
print("output:", tuple(out.shape), f"  parameters: {n_all / 1e6:.1f} M ({n_enc / 1e6:.1f} M pretrained)")
print("correct:", tuple(out.shape) == (2, NUM_CLASSES, 128, 256)
      and torch.equal(model.encoder[0][0].weight.cpu(), vgg.features[0].weight))

**Think:** the input must be divisible by $2^5 = 32$ (128 and 256 are). What would go wrong with a 100×100 image?

## 5. Train it

### Augmenting an image *and* its mask

In Lab 7 a flipped pizza was still a pizza, so only the image changed. In segmentation the label is a **picture** too:
if the photo is flipped, the label map must be flipped **in the same way**, or every pixel gets the wrong label.
(In A2 you had to do the same for the cell masks.)

**TODO 6:** write `random_flip(images, labels)`. With probability 0.5 (`random.random() < 0.5`), flip the **whole
batch** left-right with `torch.flip`. Careful: images are `(batch, 3, H, W)` but labels are `(batch, H, W)`, so the
width is a different dimension in each.

In [ ]:
def random_flip(images, labels):
    # images: (batch, 3, H, W), labels: (batch, H, W)
    # TODO: with probability 0.5, flip both left-right
    ...
    return images, labels

#### Solution

Try it yourself first, then open the hidden cell below.

In [ ]:
#@title Solution

def random_flip(images, labels):
    # images: (batch, 3, H, W), labels: (batch, H, W)
    if random.random() < 0.5:
        images = torch.flip(images, dims=[3])
        labels = torch.flip(labels, dims=[2])
    return images, labels

In [ ]:
# run this cell to check your code
images = torch.rand(4, 3, 8, 16)
labels = (images[:, 0] > 0.5).long()             # a label that "matches" the image
flipped_any = False
still_matching = True
for _ in range(20):
    im2, lab2 = random_flip(images, labels)
    flipped_any = flipped_any or not torch.equal(im2, images)
    still_matching = still_matching and torch.equal(lab2, (im2[:, 0] > 0.5).long())
print("correct:", flipped_any and still_matching and im2.shape == images.shape and lab2.shape == labels.shape)

### The training loop

This is the same loop as in Lab 7. Three things are new:

- the network gives `(batch, 19, H, W)` scores, and the labels are `(batch, H, W)`. `nn.CrossEntropyLoss` handles
  this directly: it is the Lab 5 loss, averaged over all pixels,
- `ignore_index=255` skips the unlabeled pixels,
- we report **mIoU** on the validation set after each epoch, using your `confusion_matrix` and `metrics`.

We feed the images as `uint8` and turn them into normalized floats on the GPU (`prepare`), which is faster.

In [ ]:
mean = torch.tensor([0.485, 0.456, 0.406], device=device).view(1, 3, 1, 1)   # ImageNet, as VGG16 was trained
std = torch.tensor([0.229, 0.224, 0.225], device=device).view(1, 3, 1, 1)


def prepare(images, labels):
    # uint8 (batch, H, W, 3) -> normalized float (batch, 3, H, W) on the device
    images = images.to(device).permute(0, 3, 1, 2).float() / 255
    images = (images - mean) / std
    return images, labels.to(device).long()


train_loader = DataLoader(TensorDataset(train_images[tr_idx], train_labels[tr_idx]), batch_size=16, shuffle=True)
val_loader = DataLoader(TensorDataset(train_images[val_idx], train_labels[val_idx]), batch_size=50)
test_loader = DataLoader(TensorDataset(test_images, test_labels), batch_size=50)
criterion = nn.CrossEntropyLoss(ignore_index=IGNORE)


def evaluate(model, loader):
    model.eval()
    cm = torch.zeros(NUM_CLASSES, NUM_CLASSES, dtype=torch.long)
    with torch.no_grad():
        for images, labels in loader:
            images, labels = prepare(images, labels)
            pred = model(images).argmax(dim=1)             # the best class for every pixel
            cm += confusion_matrix(pred.cpu(), labels.cpu())
    return cm


def train_model(model, optimizer, num_epochs, criterion=criterion, path="segnet.pth"):
    best_miou = 0
    for epoch in range(num_epochs):
        model.train()
        start = time.time()
        total_loss = 0
        for images, labels in train_loader:
            images, labels = prepare(images, labels)
            images, labels = random_flip(images, labels)
            loss = criterion(model(images), labels)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            total_loss += loss.item() * len(images)
        acc, miou, _ = metrics(evaluate(model, val_loader))
        if miou > best_miou:
            best_miou = miou
            torch.save(model.state_dict(), path)
        print(f"epoch {epoch + 1} | loss {total_loss / len(tr_idx):.3f} | val pixel acc {acc:.3f} "
              f"| val mIoU {miou:.3f} | {time.time() - start:.0f} s")
    model.load_state_dict(torch.load(path))
    print("best val mIoU:", round(best_miou, 3))

Now train. One epoch takes about **2 to 3 minutes** on a GPU, so start it and read the next part while you wait.

In [ ]:
torch.manual_seed(0)
random.seed(0)
model = SegNet().to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)
train_model(model, optimizer, num_epochs=5)

## 6. Results

First the one honest number on the test set, next to the two "cheating" models from Part 2:

In [ ]:
test_cm = evaluate(model, test_loader)
acc, miou, iou = metrics(test_cm)
results["SegNet (ours)"] = (acc, miou)
print(f"{'model':20s}  pixel acc   mIoU")
for name, (a, m) in results.items():
    print(f"{name:20s}  {a:9.3f}  {m:5.3f}")

In [ ]:
# photo | truth | prediction | mistakes (red = wrong, black = no label). Run again for other photos
idx = random.sample(range(len(test_images)), 4)
images, labels = prepare(test_images[idx], test_labels[idx])
model.eval()
with torch.no_grad():
    preds = model(images).argmax(dim=1).cpu()

fig, axes = plt.subplots(4, 4, figsize=(18, 6.5))
for row in range(4):
    lab = test_labels[idx[row]]
    wrong = (preds[row] != lab) & (lab != IGNORE)
    mistakes = np.full((128, 256, 3), 255, dtype=np.uint8)
    mistakes[wrong.numpy()] = [220, 0, 0]
    mistakes[(lab == IGNORE).numpy()] = 0
    panels = [test_images[idx[row]].numpy(), colorize(lab), colorize(preds[row]), mistakes]
    for col in range(4):
        axes[row, col].imshow(panels[col])
        axes[row, col].axis("off")
for ax, t in zip(axes[0], ["photo", "truth", "SegNet", "mistakes"]):
    ax.set_title(t)
plt.tight_layout()
plt.show()

**Think:** where are the red pixels? Mostly in the middle of objects, or along their **borders**?

In [ ]:
# IoU of every class on the test set, next to how common the class is
order = np.argsort(-freq)
fig, ax1 = plt.subplots(figsize=(13, 3.8))
ax1.bar(np.array(CLASSES)[order], iou.numpy()[order], color=COLORS[order] / 255, edgecolor="black")
ax1.set_ylabel("test IoU")
ax1.set_ylim(0, 1)
ax2 = ax1.twinx()
ax2.plot(np.array(CLASSES)[order], 100 * freq[order], "k.--")
ax2.set_yscale("log")
ax2.set_ylabel("% of pixels (dots)")
ax1.set_xticks(range(NUM_CLASSES))
ax1.set_xticklabels(np.array(CLASSES)[order], rotation=45, ha="right")
plt.show()

In [ ]:
# the most common mistakes: what share of class t's pixels were predicted as class p?
rows = test_cm.double() / test_cm.sum(dim=1, keepdim=True).clamp(min=1)
rows.fill_diagonal_(0)
top = torch.argsort(rows.flatten(), descending=True)[:8]
for k in top.tolist():
    t = k // NUM_CLASSES
    p = k % NUM_CLASSES
    print(f"{rows[t, p] * 100:4.0f}% of {CLASSES[t]:13s} pixels -> {CLASSES[p]}")

**Think:**
- the classes on the left of the bar chart are common **and** big. Which ones on the right have low IoU because they
  are rare, and which because they are **thin** (a pole is only a few pixels wide at 128×256)?
- are the confused pairs mistakes a human could make? Rider vs person, truck vs car, terrain vs vegetation...

**Optional experiments** (one at a time; compare the **validation** mIoU):
- **Class weights.** Give rare classes a bigger weight in the loss, e.g.
  `w = 1 / torch.log(1.02 + torch.tensor(freq))` and `nn.CrossEntropyLoss(weight=w.float().to(device), ignore_index=IGNORE)`.
  Does the mIoU go up? What happens to the pixel accuracy?
- **No pretraining.** `vgg16_bn(weights=None)` inside `vgg_stages`. How much worse is it after the same epochs?
- **More epochs**, or a learning-rate schedule from Lab 7.

In [ ]:
# our model only ever saw German streets from a car. What does it make of a street in New Jersey?
url = "https://raw.githubusercontent.com/arash-mham/visual-computing-I/main/lab-material/lab6/sopranos.jpg"
if not os.path.exists("sopranos.jpg"):
    urllib.request.urlretrieve(url, "sopranos.jpg")
photo = Image.open("sopranos.jpg").convert("RGB")
small = torch.from_numpy(np.array(photo.resize((256, 128))))[None]
images, _ = prepare(small, torch.zeros(1, 128, 256))
with torch.no_grad():
    pred = model(images).argmax(dim=1)[0].cpu()

fig, axes = plt.subplots(1, 2, figsize=(14, 3.6))
axes[0].imshow(small[0])
axes[1].imshow(colorize(pred))
for ax in axes:
    ax.axis("off")
used = torch.unique(pred).tolist()            # the classes the model predicted in this photo
legend = [Patch(color=COLORS[c] / 255, label=CLASSES[c]) for c in used]
axes[1].legend(handles=legend, bbox_to_anchor=(1.02, 1), loc="upper left")
plt.show()

**Think:** the camera is on the sidewalk, not on a car, so nothing is where the model expects it. What did it get
right, and what not? Remember the "average street" baseline: how much did the model learn about **position**?

## 7. Segmentation in the wild: portrait mode

Phones blur the background behind a person ("portrait mode"), and video calls replace it. Both need a mask of the
person. torchvision has segmentation models trained on COCO photos with the 20 classes of
[Pascal VOC](http://host.robots.ox.ac.uk/pascal/VOC/) (person, dog, car, chair, ...). We use
**DeepLabV3** ([Chen et al., 2017](https://arxiv.org/abs/1706.05587)) with a small MobileNetV3 encoder
(42 MB, it runs fine on a CPU):

In [ ]:
from torchvision.models.segmentation import deeplabv3_mobilenet_v3_large, DeepLabV3_MobileNet_V3_Large_Weights

weights = DeepLabV3_MobileNet_V3_Large_Weights.DEFAULT
deeplab = deeplabv3_mobilenet_v3_large(weights=weights).eval()
VOC = weights.meta["categories"]
print(VOC)

x = weights.transforms()(photo)[None]          # resize + normalize, as the model was trained
with torch.no_grad():
    scores = deeplab(x)["out"]                 # (1, 21, h, w)
scores = F.interpolate(scores, size=(photo.height, photo.width), mode="bilinear")   # back to the photo size
voc_pred = scores.argmax(dim=1)[0].numpy()
for c in np.unique(voc_pred):
    print(f"{VOC[c]:15s} {np.mean(voc_pred == c) * 100:5.1f} % of the photo")

**TODO 7:** write `combine(image, mask, background)`: where `mask` is True, take the pixel from `image`, everywhere
else from `background`. All three images have the same size.
Hint: `np.where(condition, a, b)`. The mask is `(H, W)` but the images are `(H, W, 3)`, so add a last axis:
`mask[:, :, None]`.

In [ ]:
def combine(image, mask, background):
    # image, background: (H, W, 3) uint8 arrays. mask: (H, W) True/False
    # TODO
    return ...

#### Solution

Try it yourself first, then open the hidden cell below.

In [ ]:
#@title Solution

def combine(image, mask, background):
    # image, background: (H, W, 3) uint8 arrays. mask: (H, W) True/False
    return np.where(mask[:, :, None], image, background)

In [ ]:
# run this cell to check your code
a = np.full((2, 3, 3), 200, dtype=np.uint8)
b = np.zeros((2, 3, 3), dtype=np.uint8)
m = np.array([[True, False, True], [False, False, True]])
out = combine(a, m, b)
print("correct:", out.shape == (2, 3, 3) and (out[m] == 200).all() and (out[~m] == 0).all())

In [ ]:
image = np.array(photo)
person = voc_pred == VOC.index("person")

blurred = np.array(photo.filter(ImageFilter.GaussianBlur(12)))
gray = np.array(photo.convert("L").convert("RGB"))
street = np.array(Image.fromarray(test_images[7].numpy()).resize(photo.size))   # a Cityscapes street

fig, axes = plt.subplots(2, 2, figsize=(16, 9.5))
panels = {
    "person mask": person,
    "portrait mode (blurred background)": combine(image, person, blurred),
    "color splash (gray background)": combine(image, person, gray),
    "new background": combine(image, person, street),
}
for ax, (name, p) in zip(axes.flatten(), panels.items()):
    ax.imshow(p, cmap="gray")
    ax.set_title(name)
    ax.axis("off")
plt.tight_layout()
plt.show()

**Think:** look at the edges of the people (hair, hands, the chair). Real portrait modes use a finer mask, and they
blur more the **farther** away something is (they also estimate depth).

### Try it yourself

Here it is a long-haired dachshund (photo: Bogdan.frus, [CC BY-SA 3.0](https://creativecommons.org/licenses/by-sa/3.0/),
Wikimedia Commons), so `target` is `"dog"`. Then upload your own photo (in Colab: the folder icon on the left), change
the file name and `target`, and run the cell again. Try a photo with a person, a dog, a cat, a car or a bike.
Which VOC classes does DeepLab find?

In [ ]:
url = "https://raw.githubusercontent.com/arash-mham/visual-computing-I/main/lab-material/lab8/dachshund.jpg"
if not os.path.exists("dachshund.jpg"):
    urllib.request.urlretrieve(url, "dachshund.jpg")

my_photo = Image.open("dachshund.jpg").convert("RGB")   # TODO (optional): your own photo
target = "dog"                                           # or "person", "cat", "car", ... (one of VOC)

with torch.no_grad():
    s = deeplab(weights.transforms()(my_photo)[None])["out"]
s = F.interpolate(s, size=(my_photo.height, my_photo.width), mode="bilinear")
mask = s.argmax(dim=1)[0].numpy() == VOC.index(target)
result = combine(np.array(my_photo), mask, np.array(my_photo.filter(ImageFilter.GaussianBlur(12))))

fig, axes = plt.subplots(1, 2, figsize=(16, 5))
axes[0].imshow(my_photo)
axes[1].imshow(result)
for ax in axes:
    ax.axis("off")
plt.show()

## Extras

### Extra A: fixing the rare classes

In Part 6 poles, traffic lights, riders and motorcycles got an IoU close to 0. The reason is the loss: cross-entropy
is an **average over pixels**. A road pixel and a pole pixel count the same, and there are 30 times more road
pixels than pole pixels, so the network learns roads first and poles hardly matter. Two common fixes:

**1. Class weights.** Multiply the loss of every pixel by a weight that is larger for rare classes. `1 / freq` would
give motorcycles a weight 400 times larger than road, which is too extreme, so ENet
([Paszke et al., 2016](https://arxiv.org/abs/1606.02147)) uses a gentler formula:

$$w_c = \frac{1}{\ln(1.02 + \text{freq}_c)}$$

**TODO 8:** compute `class_weights` (a float tensor of 19 numbers, on the device) from `freq` (the NumPy array from
Part 1), and build `weighted_ce` with `nn.CrossEntropyLoss(weight=..., ignore_index=IGNORE)`.

In [ ]:
# TODO: one weight per class (torch.log, torch.tensor(freq)), as a float tensor on the device
class_weights = ...
# TODO: cross-entropy with these weights that still ignores the unlabeled pixels
weighted_ce = ...

#### Solution

Try it yourself first, then open the hidden cell below.

In [ ]:
#@title Solution

class_weights = (1 / torch.log(1.02 + torch.tensor(freq))).float().to(device)
weighted_ce = nn.CrossEntropyLoss(weight=class_weights, ignore_index=IGNORE)

In [ ]:
# run this cell to check your code
for c in [0, 2, 5, 12, 17]:
    print(f"{CLASSES[c]:12s} freq {100 * freq[c]:5.2f} %   weight {class_weights[c].item():5.1f}")
s = torch.randn(2, NUM_CLASSES, 4, 4, device=device)
y = torch.randint(0, NUM_CLASSES, (2, 4, 4), device=device)
y[0, 0] = IGNORE
print("correct:", class_weights.shape == (NUM_CLASSES,) and class_weights.dtype == torch.float32
      and abs(class_weights[0].item() - 1 / np.log(1.02 + freq[0])) < 1e-4
      and abs(weighted_ce(s, y).item() - F.cross_entropy(s, y, weight=class_weights, ignore_index=IGNORE).item()) < 1e-5)

**2. Dice loss.** You know the Dice score from medical segmentation (A2). For one class $c$:

$$\text{Dice}_c = \frac{2\,|\text{true}_c \cap \text{pred}_c|}{|\text{true}_c| + |\text{pred}_c|}$$

To use it as a loss we need it to be differentiable, so instead of hard 0/1 masks we use the **probabilities**
$p$ (softmax) and the one-hot truth $t$ (1 at the true class of the pixel, 0 elsewhere), summed over all pixels:

$$\text{Dice}_c = \frac{2 \sum p_c\, t_c + 1}{\sum p_c + \sum t_c + 1}, \qquad
\text{loss} = 1 - \text{mean over classes of } \text{Dice}_c$$

(the $+1$ avoids $0/0$ for a class that is not in the batch). Every **class** counts the same in the mean,
no matter how many pixels it has, just like in mIoU.

**TODO 9:** finish `dice_loss`. The probabilities and the one-hot truth are ready, with the unlabeled pixels set to
0 in both. Sum over the batch and the pixels with `.sum(dim=(0, 2, 3))`, which leaves one number per class.

In [ ]:
def dice_loss(scores, labels):
    # scores: (batch, 19, H, W), labels: (batch, H, W)
    probs = F.softmax(scores, dim=1)                               # (batch, 19, H, W)
    keep = labels != IGNORE
    safe = torch.where(keep, labels, torch.zeros_like(labels))     # 255 -> 0, so one_hot works
    truth = F.one_hot(safe, NUM_CLASSES).permute(0, 3, 1, 2).float()   # (batch, 19, H, W)
    keep = keep[:, None].float()                                   # (batch, 1, H, W)
    probs = probs * keep
    truth = truth * keep
    # TODO: sum of p * t, and sum of p + sum of t, for every class (19 numbers each)
    intersection = ...
    total = ...
    # TODO: Dice of every class, then the loss
    dice = ...
    return ...

#### Solution

Try it yourself first, then open the hidden cell below.

In [ ]:
#@title Solution

def dice_loss(scores, labels):
    # scores: (batch, 19, H, W), labels: (batch, H, W)
    probs = F.softmax(scores, dim=1)                               # (batch, 19, H, W)
    keep = labels != IGNORE
    safe = torch.where(keep, labels, torch.zeros_like(labels))     # 255 -> 0, so one_hot works
    truth = F.one_hot(safe, NUM_CLASSES).permute(0, 3, 1, 2).float()   # (batch, 19, H, W)
    keep = keep[:, None].float()                                   # (batch, 1, H, W)
    probs = probs * keep
    truth = truth * keep
    intersection = (probs * truth).sum(dim=(0, 2, 3))
    total = probs.sum(dim=(0, 2, 3)) + truth.sum(dim=(0, 2, 3))
    dice = (2 * intersection + 1) / (total + 1)
    return 1 - dice.mean()

In [ ]:
# run this cell to check your code
y = torch.randint(0, NUM_CLASSES, (2, 16, 16), device=device)
perfect = 50 * F.one_hot(y, NUM_CLASSES).permute(0, 3, 1, 2).float()      # very sure, and right
wrong = 50 * F.one_hot((y + 1) % NUM_CLASSES, NUM_CLASSES).permute(0, 3, 1, 2).float()   # very sure, and wrong
y_ign = y.clone()
y_ign[:, :8] = IGNORE
half_wrong = perfect.clone()
half_wrong[:, :, :8] = wrong[:, :, :8]                                    # wrong only where there is no label
print(f"perfect {dice_loss(perfect, y).item():.3f}   wrong {dice_loss(wrong, y).item():.3f}   "
      f"wrong only on unlabeled pixels {dice_loss(half_wrong, y_ign).item():.3f}")
print("correct:", dice_loss(perfect, y).item() < 0.01 and dice_loss(wrong, y).item() > 0.95
      and dice_loss(half_wrong, y_ign).item() < 0.01)

Now train a second SegNet, exactly like in Part 5 but with the loss **weighted cross-entropy + Dice**
(about the same time as before). Then compare it with the first model on the test set.

In [ ]:
def combined_loss(scores, labels):
    return weighted_ce(scores, labels) + dice_loss(scores, labels)


torch.manual_seed(0)
random.seed(0)
model2 = SegNet().to(device)
optimizer = torch.optim.Adam(model2.parameters(), lr=1e-4)
train_model(model2, optimizer, num_epochs=5, criterion=combined_loss, path="segnet_rare.pth")

In [ ]:
_, _, iou_first = metrics(test_cm)
test_cm2 = evaluate(model2, test_loader)
acc2, miou2, iou_second = metrics(test_cm2)
results["SegNet, weighted CE + Dice"] = (acc2, miou2)
print(f"{'model':28s}  pixel acc   mIoU")
for name, (a, m) in results.items():
    print(f"{name:28s}  {a:9.3f}  {m:5.3f}")

x = np.arange(NUM_CLASSES)
plt.figure(figsize=(14, 3.8))
plt.bar(x - 0.2, iou_first.numpy()[order], width=0.4, label="cross-entropy (Part 5)")
plt.bar(x + 0.2, iou_second.numpy()[order], width=0.4, label="weighted cross-entropy + Dice")
plt.xticks(x, np.array(CLASSES)[order], rotation=45, ha="right")
plt.ylabel("test IoU")
plt.ylim(0, 1)
plt.legend()
plt.show()

**Think:** which classes gained the most? Did the pixel accuracy go **down** a little? (The network now predicts
rare classes more often, so it sometimes "sees" a pole where there is a building.) Which number would matter more
for a self-driving car?

To find out which fix helped more, train once with only `weighted_ce` and once with only `dice_loss`.

### Extra B: how many cars?

Our model says which pixels are **car**, but not how many cars there are. A simple idea: every connected blob of car
pixels is one car. Finding the blobs is called **connected-component labeling**, and the classic algorithm is
**flood fill** (the paint-bucket tool in Paint):

```
for every pixel:
    if it is a car pixel that has no number yet:
        new number; start a stack with this pixel
        while the stack is not empty:
            take a pixel from the stack
            every neighbor (up, down, left, right) that is a car pixel without a number:
                give it the same number and put it on the stack
```

**TODO 10:** finish the inner loop of `connected_components`. For each of the 4 neighbors `(ny, nx)`: check that it
is **inside** the image, that it is in the mask, and that it has no number yet (`ids[ny, nx] == 0`). If so, give it
the number `count` and push it on the stack.

In [ ]:
def connected_components(mask):
    # mask: (H, W) True/False. Returns ids (H, W) with 0 = background and 1, 2, ... = blob number, and the count
    H, W = mask.shape
    ids = np.zeros((H, W), dtype=int)
    count = 0
    for y in range(H):
        for x in range(W):
            if mask[y, x] and ids[y, x] == 0:       # a new blob starts here
                count += 1
                ids[y, x] = count
                stack = [(y, x)]
                while len(stack) > 0:
                    cy, cx = stack.pop()
                    for ny, nx in [(cy - 1, cx), (cy + 1, cx), (cy, cx - 1), (cy, cx + 1)]:
                        # TODO: inside the image, in the mask, and no number yet -> same number, onto the stack
                        ...
    return ids, count

#### Solution

Try it yourself first, then open the hidden cell below.

In [ ]:
#@title Solution

def connected_components(mask):
    # mask: (H, W) True/False. Returns ids (H, W) with 0 = background and 1, 2, ... = blob number, and the count
    H, W = mask.shape
    ids = np.zeros((H, W), dtype=int)
    count = 0
    for y in range(H):
        for x in range(W):
            if mask[y, x] and ids[y, x] == 0:       # a new blob starts here
                count += 1
                ids[y, x] = count
                stack = [(y, x)]
                while len(stack) > 0:
                    cy, cx = stack.pop()
                    for ny, nx in [(cy - 1, cx), (cy + 1, cx), (cy, cx - 1), (cy, cx + 1)]:
                        if 0 <= ny < H and 0 <= nx < W and mask[ny, nx] and ids[ny, nx] == 0:
                            ids[ny, nx] = count
                            stack.append((ny, nx))
    return ids, count

In [ ]:
# run this cell to check your code
toy = np.array([[1, 1, 0, 0, 1],
                [0, 1, 0, 1, 1],
                [0, 0, 0, 0, 0],
                [1, 0, 1, 1, 0]], dtype=bool)
ids, count = connected_components(toy)
print(ids)
print("correct:", count == 4 and ids[0, 0] == ids[1, 1] != ids[0, 4] and ids[0, 4] == ids[1, 3]
      and ids[3, 2] == ids[3, 3] and (ids[~toy] == 0).all() and len(np.unique(ids)) == 5)

In [ ]:
# cars in the truth and in our prediction. Blobs smaller than 20 pixels are ignored (noise)
car = CLASSES.index("car")
i = 1          # try other test images
images, labels = prepare(test_images[i:i + 1], test_labels[i:i + 1])
with torch.no_grad():
    pred = model(images).argmax(dim=1)[0].cpu().numpy()
truth = test_labels[i].numpy()
pred[truth == IGNORE] = IGNORE      # the camera car's own hood has no label, so skip it in the prediction too


def blobs(mask, min_size=20):
    ids, count = connected_components(mask)
    for k in range(1, count + 1):
        if (ids == k).sum() < min_size:
            ids[ids == k] = 0
    ids_left = np.unique(ids[ids > 0])
    return ids, len(ids_left)


rng = np.random.default_rng(0)
random_colors = rng.integers(50, 256, size=(128 * 256 + 1, 3)).astype(np.uint8)
random_colors[0] = 0
fig, axes = plt.subplots(1, 3, figsize=(18, 3.2))
axes[0].imshow(test_images[i])
axes[0].set_title("photo")
for ax, (name, lab) in zip(axes[1:], [("truth", truth), ("SegNet", pred)]):
    ids, n = blobs(lab == car)
    ax.imshow(random_colors[ids])
    ax.set_title(f"{name}: {n} car blobs")
for ax in axes:
    ax.axis("off")
plt.show()

**Think:** count the cars in the photo yourself. Parked cars that touch each other become **one** blob, even in the
**true** label map: a semantic label only says "car", not *which* car. To count objects we need **instance
segmentation**, where every object gets its own mask. That is the topic of Lab 10.